# 🔗 07. Multi-Modal Late Fusion Training & Calibration
Huấn luyện và chuẩn hóa trọng số kết hợp giữa điểm số động học (Motion) và điểm số thị giác (Visual).

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from src.fusion.fusion import AccidentFusion

fusion_model = AccidentFusion(method="mlp")
print("Fusion Module initialized:", fusion_model)

## 1. Huấn luyện MLP Fusion trên Validation Scores

In [ ]:
# Giả lập tập điểm số từ validation
N = 200
true_labels = np.random.choice([0.0, 1.0], size=(N, 1), p=[0.7, 0.3]).astype(np.float32)
motion_scores = np.clip(true_labels * 0.8 + np.random.normal(0, 0.2, (N, 1)), 0.0, 1.0).astype(np.float32)
visual_scores = np.clip(true_labels * 0.9 + np.random.normal(0, 0.15, (N, 1)), 0.0, 1.0).astype(np.float32)

m_t = torch.tensor(motion_scores)
v_t = torch.tensor(visual_scores)
y_t = torch.tensor(true_labels)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(fusion_model.parameters(), lr=0.01)

for epoch in range(50):
    optimizer.zero_grad()
    preds = fusion_model(m_t, v_t)
    loss = criterion(preds, y_t)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/50 Loss: {loss.item():.4f}")

fusion_model.save_checkpoint("checkpoints/fusion_best.pt")
print("✅ Saved Fusion best checkpoint!")